In [36]:
import numpy as np
from gates import (Z, X, Y, H, CNOT, CZ, P, T, S, applyGate, kron)
from functools import reduce

# basics

In [9]:
s0 = np.array([1,0])
s1 = np.array([0,1])
sp = (1/np.sqrt(2))*np.array([1,1])
sm = (1/np.sqrt(2))*np.array([1,-1])
sT = (1/np.sqrt(2)) * (s0 + np.exp(1j*np.pi/4)*s1)

(
    applyGate(H, s0) == sp, 
    applyGate(H, s1) == sm, 
    applyGate(H, sp) == s0, 
    applyGate(H, sm) == s1,
    
    applyGate(X, s0) == s1,
    applyGate(X, s1) == s0,
    applyGate(X, sm) == -sm,
    applyGate(X, sp) == sp,
    
    )

(array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]),
 array([ True,  True]))

# Apply X, Y, Z, H, and T to $\ket{0}$, $\ket{1}$, $\ket{+}$, $\ket{T} = (\ket{0}+e^{i\pi/4}\ket{1})/\sqrt{2}$

In [10]:
results = (
    applyGate(X, s0) == s1,
    applyGate(X, s1) == s0,
    applyGate(X, sp) == sp,
    applyGate(X, sm) == -sm,
    applyGate(X, sT),
    "------------------",
    applyGate(Y, s0) == 1j*s1,
    applyGate(Y, s1) == -1j*s0,
    applyGate(Y, sp) == -1j*sm,
    applyGate(Y, sm) == 1j*sp,
    applyGate(Y, sT),
    "------------------",
    applyGate(Z, s0) == s0,
    applyGate(Z, s1) == -s1,
    applyGate(Z, sp) == sm,
    applyGate(Z, sm) == sp,
    applyGate(Z, sT),
    "------------------",
    applyGate(H, s0) == sp,
    applyGate(H, s1) == sm,
    applyGate(H, sp) == s0,
    applyGate(H, sm) == s1,
    applyGate(H, sT),
    
)

with np.printoptions(precision=3, suppress=True):
    for res in results:
        print(res)
        # res

[ True  True]
[ True  True]
[ True  True]
[ True  True]
[0.5  +0.5j 0.707+0.j ]
------------------
[ True  True]
[ True  True]
[ True  True]
[ True  True]
[0.5-0.5j   0. +0.707j]
------------------
[ True  True]
[ True  True]
[ True  True]
[ True  True]
[ 0.707+0.j  -0.5  -0.5j]
------------------
[ True  True]
[ True  True]
[ True  True]
[ True  True]
[0.854+0.354j 0.146-0.354j]


# 2qubit states and CNOT, CZ

In [11]:
s00 = kron(s0, s0)
s01 = kron(s0, s1)
s10 = kron(s1, s0)
s11 = kron(s1, s1)
sp0 = kron(sp, s0)
spp = kron(sp, sp)

results = (
    "CNOT",
    applyGate(CNOT, s00) == s00,
    applyGate(CNOT, s01) == s01,
    applyGate(CNOT, s11) == s10,
    applyGate(CNOT, sp0) == (s00 + s11)/np.sqrt(2),
    applyGate(CNOT, spp) == spp,
    "------------------",
    "CZ",
    applyGate(CZ, s00) ==  s00,
    applyGate(CZ, s01) ==  s01,
    applyGate(CZ, s10) == s10,
    applyGate(CZ, s11) == -s11,
    applyGate(CZ, sp0) == sp0,
    np.isclose(applyGate(CZ, spp), (1/2) * (s00 + s01 + s10 - s11))
)

for res in results:
    print(res)

CNOT
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
------------------
CZ
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]
[ True  True  True  True]


# Implement quantum teleportation register

Given the system  

![Quantum teleportation circuit](https://upload.wikimedia.org/wikipedia/commons/thumb/d/dc/Quantum_teleportation_circuit.svg/1920px-Quantum_teleportation_circuit.svg.png)

We want to determine the operations needed for the system.





$$
    (M\otimes CNOT)(I\otimes M\otimes I)(H\otimes M\otimes I)(CNOT\otimes I)\ket{\psi}_A\ket{\phi}_A^+\ket{\phi}_B^+
$$

In [12]:
I = np.eye(2)

s000 = kron(s0, s0, s0); print(f"{s000 = }")
s001 = kron(s0, s0, s1); print(f"{s001 = }")
s010 = kron(s0, s1, s0); print(f"{s010 = }")
s011 = kron(s0, s1, s1); print(f"{s011 = }")
s100 = kron(s1, s0, s0); print(f"{s100 = }")
s101 = kron(s1, s0, s1); print(f"{s101 = }")
s110 = kron(s1, s1, s0); print(f"{s110 = }")
s111 = kron(s1, s1, s1); print(f"{s111 = }")
print(kron(CNOT, I))
np.all(kron(CNOT, I) @ s110 == s100)

s000 = array([1, 0, 0, 0, 0, 0, 0, 0])
s001 = array([0, 1, 0, 0, 0, 0, 0, 0])
s010 = array([0, 0, 1, 0, 0, 0, 0, 0])
s011 = array([0, 0, 0, 1, 0, 0, 0, 0])
s100 = array([0, 0, 0, 0, 1, 0, 0, 0])
s101 = array([0, 0, 0, 0, 0, 1, 0, 0])
s110 = array([0, 0, 0, 0, 0, 0, 1, 0])
s111 = array([0, 0, 0, 0, 0, 0, 0, 1])
[[1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]]


np.True_

In [ ]:

def baseGate(N):
    I = np.eye(2)
    return [I] * N

def singleGate(gate : np.ndarray, target : int, N : int) -> np.ndarray:
    ops = baseGate(N)
    ops[target] = gate
    ops = reduce(np.kron, ops)
    return ops
    
    
def controlGate(gate : np.ndarray, control : int, target : int, N : int) -> np.ndarray:
    # if isinstance(controls, int):
    #     controls = [controls]
    assert target != control, "Control and target cannot be the same"
    
    
    s0 = np.array([[1,0]]).T # |0>
    s1 = np.array([[0,1]]).T # |1>
    proj0 = s0@s0.T  # |0><0|
    proj1 = s1@s1.T  # |1><1|
    
    ops1 = baseGate(N)
    ops2 = baseGate(N)
    
    ops1[control] = proj0
    ops2[control] = proj1
        
    ops2[target] = gate
    
    ops1 = reduce(np.kron, ops1)
    ops2 = reduce(np.kron, ops2)
    
    return ops1 + ops2


def measureGate(states : list, target : int) -> int:
    
    state = states[target]
    rng = np.random.default_rng()
    probabilities = np.abs(state)**2
    return rng.choice([0,1], p=probabilities)

def controlMeasure(gate, target, measurement, N):
    
    ops = baseGate(N)
    # ops2 = [I]*N
    
    if measurement == 1:
        ops[target] = gate
        
    return reduce(np.kron, ops)

def get_channel(fullState : np.ndarray, qubits_per_channel : tuple):
    
    channel_state = []
    for i, n in enumerate(qubits_per_channel):
        channel_state.append(fullState[i*n:(i+1)*n])
    
    

In [ ]:
inputState = np.array([1, 0])
bellState = (kron(s0,s0) + kron(s1, s1)) / np.sqrt(2)
states_list = [inputState, bellState, bellState]
inputFull = reduce(np.kron, states_list)
N = int( np.log2(len(inputFull)) ) # number of qubits


state = controlGate(X, 0, 1, N) @ inputFull # CNOT
print(state)
state = singleGate(H, 0, N) @ state   # Hadamard
print(state)
state = 



[0.5 0.  0.  0.5 0.  0.  0.  0.  0.  0.  0.  0.  0.5 0.  0.  0.5 0.  0.
 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
[0.35355339 0.         0.         0.35355339 0.         0.
 0.         0.         0.         0.         0.         0.
 0.35355339 0.         0.         0.35355339 0.35355339 0.
 0.         0.35355339 0.         0.         0.         0.
 0.         0.         0.         0.         0.35355339 0.
 0.         0.35355339]


In [302]:
testState = np.asarray(
    kron(s1, s0, s1), 
    dtype=np.float64)
print(testState)
print(
    controlGate(gate=X, control=0, target=2, N=3) @ testState
)
print(np.asarray(kron(s1,s0,s0), dtype=np.float64))

[0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0.]


In [ ]:
state = kron(s0, s0, s0)

controlGate(X, 0, 1, 3) @ state


array([1., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
# class Qubit:
#     def __init__(self, vector):
#         norm = np.linalg.norm(vector)
#         assert np.isclose(norm, 1.0), "Qubit must be normalized"
#         self.vector = np.asarray(vector, dtype=np.float64)
#         self.rng = np.random.default_rng()
#         self.ket = self._make_ket()
    
#     def _make_ket(self):
#         if self.vector.shape == (2,):
#             if np.all(self.vector == [1, 0]): return "|0>"
#             elif np.all(self.vector == [0, 1]): return "|1>"
#             elif np.all(self.vector == np.array([1, 1])/np.sqrt(2)): return "|+>"
#             elif np.all(self.vector == np.array([1, -1])/np.sqrt(2)): return "|->"
#             pass # no easy qubit ket
#         else: return "" # no easily determined ket
    
#     def measure(self, basis = "Z"):
        
#         if basis == "Z": # ensure Z-basis
#             basis = (np.array([1, 0]),
#                      np.array([0, 1]))
#         elif basis == "X":
#             raise ValueError("Not implemented!")
#             basis = (np.array([0, 1]),
#                      np.array([1, 0]))
#         else:
#             raise ValueError(f"'basis' must be 'Z' (default) or 'X', but was {basis}")
            
#         prob_0 = np.abs(np.vdot(basis[0], self.vector))**2 # || <0|psi> ||^2
#         prob_1 = np.abs(np.vdot(basis[1], self.vector))**2 # || <1|psi> ||^2
        
#         probabilities = np.array([prob_0, prob_1]) 
#         probabilities /= probabilities.sum()
#         return self.rng.choice([0, 1], p=probabilities) # return 0 or 1 with probabilities
    
#     def __repr__(self): # How does it look when printing the object
#         return f"Qubit(state = {self.vector}; {self.ket})"
    
# A = Qubit(s0)

# print(A)
# print(A.measure())


# class Teleportation:
    
#     def __init__(self, qubit_in, *, DEBUG=False):
#         # self.qubit = qubit_in
        
#         if DEBUG:
#             bell_state = np.array([1, 0])
#         else:
#             bell_state = self._create_bell_state()
        
        
#         self.bell_state = Qubit(bell_state)
#         self.qubits = [qubit_in.vector, self.bell_state.vector, self.bell_state.vector]
#         print(self.qubits)
#         self.full_state = reduce(np.kron, self.qubits)
#         self.total_dim = len(self.full_state) 
#         self.num_qubits = int(np.log2(self.total_dim)) # number of qubits -- find N from a 2**N length array
        
#     def _create_bell_state(self):
#         s0 = np.array([1,0])
#         s1 = np.array([0,1])
#         bell = (kron(s0, s0) + kron(s1, s1)) / np.sqrt(2)
#         return bell
    
#     def applyGate(self, gate, target):
#         # I = np.eye(2)
#         # if gate.shape[0] == 2:
#         #     # single qubit gate
#         #     ops = [gate if i == target else I for i in range(self.num_qubits)]
#         # elif gate.shape[0] == 4:
#         #     # two qubit gate
#         #     assert len(self.qubit.state) == 2, "Two-qubit gate needs two-target indices"
#         #     ops = []
#         #     for k in range(self.num_qubits):
#         #         if k in target:
#         #             ops.append(gate)
                    
                    
            
        
#         # self.full_state = gate @ self.full_state
#         # return self.full_state
    
#     # def teleport(self):
        
#     #     # step 1
#     #     Gate = kron(CNOT, I)
#     #     self.applyGate(Gate)
        
#     #     # Step 2 
#     #     Gate = kron(H, I, I)
#     #     self.applyGate(Gate)
        
#     #     # Step 3 
        
        
        
    
#     def __repr__(self):
#         with np.printoptions(formatter={"all": lambda x: f"{x:>5.2f},"}, linewidth=110):
#             return f"Teleportation(\n" \
#                 f"\t        input={self.qubit.vector};  {self.qubit.ket},\n"\
#                 f"\t{int(self.num_qubits)}-Qubit state=\n{self.full_state}\n  )"
        

# test = Teleportation(Qubit(s1), DEBUG=False)

# with np.printoptions(formatter={"all": lambda x: f"{x:>6.2f},"}, linewidth=110):
#     # print(test)
#     # print(test.full_state)
#     # print(test.applyGate(kron(CNOT, I)))
#     print()

Qubit(state = [1. 0.]; |0>)
0
